# 多模态
## 1. 什么是多模态
多模态是指模型能够同时理解和处理 不止一种类型的数据（模态），例如：
- 文本（Text）
- 图像（Vision）
- 音频（Audio）
- 视频（Video）
- 传感器数据（Sensor / IoT 数据）

人类在感知和交流时是天然的多模态，比如我们读书时同时看到文字和插图，听别人讲话还能看到对方的表情。

## 2. 传统模型
传统的 AI 模型往往只处理单一模态：
- NLP 模型 只会处理文本。
- CV 模型 只会处理图像。
- ASR/TTS 模型 只会处理音频。

而多模态模型可以融合多种输入信息，实现更强的理解和生成能力。
例如：
- 图文混合理解：问模型“这张图片中有几只猫？”
- 文生图：输入文字提示，生成符合语义的图片（Stable Diffusion）
- 看图回答问题：上传一张图，提问并得到基于图片内容的答案
- 视频理解：给视频自动生成摘要或标签
- 音视频分析：例如自动识别视频中的语音并关联画面内容

## 3. 任务分类
1. Audio-Text-to-Text (音频-文本到文本)：将音频（如语音）和辅助文本结合作为输入，生成文本输出。例如：语音转录时结合上下文文本提升准确性。
2. Image-Text-to-Text (图像-文本到文本)：结合图像和文本输入生成文本输出。例如：图像描述生成（输入图片+提示文本，输出详细描述）。
3. Visual Question Answering (视觉问答，VQA)：对给定的图像和自然语言问题生成答案。例如：输入图片+“这是什么动物？”，输出“狗”。
4. Document Question Answering (文档问答)：基于文档（如PDF、扫描件）中的文本或布局信息回答问题。例如：从发票中提取金额或日期。
5. Video-Text-to-Text (视频-文本到文本)：结合视频帧序列和文本输入生成文本输出。例如：视频内容摘要或基于视频的对话生成。
6. Visual Document Retrieval (视觉文档检索)：通过图像或文本查询搜索相关文档。例如：上传表格图片，检索数据库中的匹配文档。
7. Any-to-Any (任意模态到任意模态)：支持任意输入和输出模态的通用模型。例如：输入音频+图片，输出文本；或输入文本，生成图片+语音。

## 4. 常用模型
| 任务 | 输入 | 	输出 |	代表模型 |
| --- | --- | --- | --- |
| 图像描述 (Image Captioning)|	图像|	文字描述|	BLIP、OFA|
|视觉问答 (Visual Question Answering, VQA)|	图像 + 文字问题|	文字答案|	LLaVA、MiniGPT-4|
|文生图 (Text-to-Image)|	文字|	图像	|DALL·E 3、Stable Diffusion|
|图像搜索 (Text-Image Retrieval)|	文字或图像	|相似图像或文本|	CLIP|
|视频理解|	视频|	文本摘要或分类标签|	VideoCLIP、TimeSformer|
|音视频多模态|	视频 + 音频|	多任务输出（转写+理解）	|Whisper + Vision Transformer|

# 1. 视觉问答（Visual Question Answering, VQA）

In [2]:
from transformers import pipeline
import requests
from PIL import Image


# 创建视觉问答pipeline; 默认使用 dandelin/vilt-b32-finetuned-vqa
vqa = pipeline("visual-question-answering")

# 定义问题和图像
image = Image.open("img/object.jpg")
question= "How many cats are there?"

# 进行视觉问答
answer = vqa(image=image, question=question)
print(answer)

No model was supplied, defaulted to dandelin/vilt-b32-finetuned-vqa and revision d0a1f6a (https://huggingface.co/dandelin/vilt-b32-finetuned-vqa).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cuda:0


[{'score': 0.8799368739128113, 'answer': '2'}, {'score': 0.295978844165802, 'answer': '1'}, {'score': 0.004855958744883537, 'answer': '3'}, {'score': 0.0034532591234892607, 'answer': '0'}, {'score': 0.000804933428298682, 'answer': 'yes'}]


# 2. 文档问答（Document Question Answering）
> 提前安装  PIL, pytesseract,  PyTorch, transformers

> 安装 pytesseract Python 包：`pip install pytesseract`

> 安装 Tesseract OCR 引擎：`apt-get install tesseract-ocr`

安装完成后要重启 jupyter lab 的python内核。才可以测试生效

`OCR 全称是 Optical Character Recognition（光学字符识别），是一种将图像中的文字转换为可编辑文本的技术。`

## 1. 测试ocr功能

In [6]:
import pytesseract
from PIL import Image

# 测试 OCR
image = Image.open("img/invoice.png")
text = pytesseract.image_to_string(image)
print("OCR 提取的文本:", text)


OCR 提取的文本: INVOICE

East Repair Inc.
1912 Harvest Lane
New York, NY 12210

 

 

BILLTO SHIP TO INVOICE # us-001
John Smith John Smith INVOICE DATE 1110212019
2 Court Square 3787 Pineview Drive Pose

New York, NY 12210 Cambridge, MA 12210 oy 2312/2019
DUE DATE 26/02/2019

ay DESCRIPTION UNIT PRICE ‘AMOUNT

1 Front and rear brake cables 100.00 100.00

2 Newset of pedal arms 15.00 30.00

3 Labor Shrs 5.00 1.00

Subtotal 145.00

Sales Tax 6.25% 9.06

TOTAL $154.06

Smith

TERMS & CONDITIONS

Payment is due within 15 days
J hank you Plate ate cack payable: Est Rep



## 2. 测试文档问答

In [7]:
from transformers import pipeline
from PIL import Image

# 如果在 Windows 上，可能需要指定 tesseract 的路径
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

nlp = pipeline(
    "document-question-answering",
    model="impira/layoutlm-document-qa",
)

image = Image.open("img/invoice.png")

res = nlp(
    image=image,
    question="What is the invoice number?"
)

print(res)

Device set to use cuda:0


[{'score': 0.4251859486103058, 'answer': 'us-001', 'start': 16, 'end': 16}]
